# Batch gradient descent for multiple linear regression

Diabetes data, 10 inputs.

In [1]:
import time
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [2]:
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
reg = LinearRegression().fit(X_train, y_train)
print("OLS test R2", r2_score(y_test, reg.predict(X_test)))
print(reg.coef_.round(1), round(reg.intercept_, 2))

OLS test R2 0.4399338661568968
[  -9.2 -205.5  516.7  340.6 -895.6  561.2  153.9  126.7  861.1   52.4] 151.88


## Our own batch gradient descent

In [3]:
class GDRegressor:
    def __init__(self, learning_rate=0.01, epochs=100):
        self.lr, self.epochs = learning_rate, epochs

    def fit(self, X, y):
        n = X.shape[0]
        self.intercept_ = 0.0
        self.coef_ = np.ones(X.shape[1])           # start: all coefficients 1
        for _ in range(self.epochs):
            y_hat = X @ self.coef_ + self.intercept_    # predictions for ALL rows
            err = y - y_hat
            intercept_der = -2 * err.mean()           # dL/d(beta_0)
            coef_der = -2 * (X.T @ err) / n           # all m derivatives at once
            self.intercept_ -= self.lr * intercept_der
            self.coef_ -= self.lr * coef_der
        return self

    def predict(self, X):
        return X @ self.coef_ + self.intercept_

start = time.perf_counter()
gdr = GDRegressor(learning_rate=0.5, epochs=1000).fit(X_train, y_train)
print("seconds", round(time.perf_counter() - start, 3))
print("GD test R2", r2_score(y_test, gdr.predict(X_test)))
print(gdr.coef_.round(1), round(gdr.intercept_, 2))

seconds 0.06
GD test R2 0.4534503034722803
[  14.4 -173.7  491.5  323.9  -39.3 -116.  -194.   103.4  451.6   97.6] 152.01


## More epochs: gradient descent moves towards the OLS coefficients

In [4]:
for epochs in [100, 1000, 10000, 50000]:
    g = GDRegressor(learning_rate=0.5, epochs=epochs).fit(X_train, y_train)
    print(epochs, "train R2", round(r2_score(y_train, g.predict(X_train)), 4),
          "test R2", round(r2_score(y_test, g.predict(X_test)), 4),
          "largest gap to OLS coef", round(np.abs(g.coef_ - reg.coef_).max(), 1))

100 train R2 0.3429 test R2 0.3202 largest gap to OLS coef 936.6


1000 train R2 0.5238 test R2 0.4535 largest gap to OLS coef 856.2


10000 train R2 0.5289 test R2 0.4429 largest gap to OLS coef 715.0


50000 train R2 0.5317 test R2 0.4418 largest gap to OLS coef 319.3


## Why vectorise: the same derivative with a Python loop

In [5]:
def coef_der_loop(X, err):
    n, m = X.shape
    der = np.zeros(m)
    for j in range(m):              # every coefficient
        total = 0.0
        for i in range(n):          # every row
            total += err[i] * X[i, j]
        der[j] = -2 * total / n
    return der

err = y_train - (X_train @ np.ones(10))
t = time.perf_counter(); d1 = coef_der_loop(X_train, err); t_loop = time.perf_counter() - t
t = time.perf_counter(); d2 = -2 * X_train.T @ err / len(err); t_vec = time.perf_counter() - t
print(np.allclose(d1, d2), f"loop {t_loop*1000:.2f} ms, vectorised {t_vec*1000:.3f} ms")

True loop 9.69 ms, vectorised 0.163 ms


## Early stopping: when does it beat OLS?

We start every coefficient at zero, choose the stopping epoch on a validation part of the training data, retrain for that many epochs and score on the test set. We repeat over 50 random splits and report the average.

In [6]:
# Test 1: when does early stopping beat OLS?  Averaged over 50 random train/test splits.
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

def gd_curve(Xa, ya, Xb, yb, epochs=2000, lr=0.04):
    """Batch gradient descent from all-zero coefficients; test MSE after every epoch."""
    w, b, mse = np.zeros(Xa.shape[1]), 0.0, []
    for _ in range(epochs):
        err = ya - (Xa @ w + b)
        b -= lr * (-2 * err.mean())
        w -= lr * (-2 * Xa.T @ err / len(Xa))
        mse.append(np.mean((yb - (Xb @ w + b)) ** 2))
    return np.array(mse), w

def early_stopping_vs_ols(X, y, n_train=200, splits=50):
    rows = []
    for s in range(splits):
        Xa, Xb, ya, yb = train_test_split(X, y, train_size=n_train, random_state=s)
        sc = StandardScaler().fit(Xa); Xa, Xb = sc.transform(Xa), sc.transform(Xb)
        ols = LinearRegression().fit(Xa, ya)
        # choose the stopping epoch on a validation part of the training data only
        Xf, Xv, yf, yv = train_test_split(Xa, ya, test_size=0.25, random_state=100 + s)
        stop = gd_curve(Xf, yf, Xv, yv)[0].argmin() + 1
        # retrain on the whole training set for that many epochs, then score on the test set
        mse, w = gd_curve(Xa, ya, Xb, yb, epochs=stop)
        rows.append((r2_score(yb, ols.predict(Xb)), 1 - mse[-1] / np.var(yb), stop,
                     np.linalg.norm(w), np.linalg.norm(ols.coef_)))
    r = np.array(rows)
    print(f"{X.shape[1]:>2} features, {n_train} training observations: OLS test R2 {r[:,0].mean():.3f}, "
          f"early stopping {r[:,1].mean():.3f}, better in {(r[:,1] > r[:,0]).sum()}/{splits} splits, "
          f"median stop epoch {np.median(r[:,2]):.0f}, coefficient length {np.median(r[:,3]):.0f} vs OLS {np.median(r[:,4]):.0f}")

X_poly = PolynomialFeatures(2, include_bias=False).fit_transform(X)   # 10 features + squares + products = 65
early_stopping_vs_ols(X, y)              # few features per observation
early_stopping_vs_ols(X_poly, y)         # many features per observation
early_stopping_vs_ols(X_poly, y, n_train=350)


10 features, 200 training observations: OLS test R2 0.468, early stopping 0.467, better in 26/50 splits, median stop epoch 58, coefficient length 43 vs OLS 67


65 features, 200 training observations: OLS test R2 0.063, early stopping 0.401, better in 50/50 splits, median stop epoch 65, coefficient length 49 vs OLS 2968


65 features, 350 training observations: OLS test R2 0.332, early stopping 0.432, better in 48/50 splits, median stop epoch 54, coefficient length 44 vs OLS 1195


In [7]:
# Test 2: is the slow direction of the loss along s1 and s2?
print("corr(s1, s2)", round(np.corrcoef(X_train[:, 4], X_train[:, 5])[0, 1], 3))
Xc = X_train - X_train.mean(0)
eigval, eigvec = np.linalg.eigh(Xc.T @ Xc / len(Xc))     # curvature of the loss in each direction
print("largest / smallest curvature", round(eigval[-1] / eigval[0]))
print("flattest direction", eigvec[:, 0].round(2))       # columns: age sex bmi bp s1 s2 s3 s4 s5 s6

corr(s1, s2) 0.895
largest / smallest curvature 447
flattest direction [ 0.    0.01  0.01 -0.    0.71 -0.56 -0.32 -0.09 -0.27  0.  ]
